In [1]:
from __future__ import print_function, division
import pandas as pd
import torch.optim as optim
import torch.nn as nn
import os
import torch
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from skimage import io
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import albumentations as A
import cv2
plt.ion()   # interactive mode

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# CUDA for PyTorch
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
torch.backends.cudnn.benchmark = True

%cd /kaggle/input/effnetfolder/EfficientNet-PyTorch


model_full_name = 'efficientnet-b4-e10'
model_name = 'efficientnet-b4'
folder_name = 'effnetmodelb4'
from efficientnet_pytorch import EfficientNet
%cd /kaggle/working

[Errno 2] No such file or directory: '/kaggle/input/effnetfolder/EfficientNet-PyTorch'
/kaggle/working


ModuleNotFoundError: No module named 'efficientnet_pytorch'

In [2]:
# Augmentation # define new augmentation here
class ToTensor(object):
    def __call__(self, image, force_apply=True):
        output = image.transpose((2, 0, 1))
        return torch.from_numpy(output)


In [3]:
class TestDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.images = os.listdir(root_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = os.path.join(self.root_dir, self.images[idx])
        image = io.imread(img_path)
        
        if self.transform:
            image = self.transform(image = image)

        return self.images[idx], image

In [4]:
transform = A.Compose([
    A.CenterCrop(width=512, height=512),
    A.Flip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, p=1.0),
    ToTensor()
])

test_image = TestDataset(root_dir = '../input/cassava-leaf-disease-classification/test_images/', transform = transform)
testloader = DataLoader(test_image, batch_size = 4, shuffle=False, num_workers=1)

# load model
model = EfficientNet.from_name(model_name, num_classes = 5).to(device)

PATH = '../input/' + folder_name + '/' + model_full_name + '.pt'
model.load_state_dict(torch.load(PATH))
model.eval()

names = []
predicted = []
for names_batch, images_batch in testloader:
    images_batch = images_batch.to(device).float()
    output = model(images_batch)
    output = torch.max(output, 1)[1].cpu().detach().numpy()
    names.extend(list(names_batch))
    predicted.extend(output)

AttributeError: module 'albumentations' has no attribute 'Flip'

In [5]:
# export 
result = pd.DataFrame(list(zip(names, predicted)), columns = ['image_id', 'label'])

result.to_csv('submission.csv', index=False)

NameError: name 'names' is not defined